In [1]:
import re
from functools import partial
from dataclasses import dataclass
from typing import Callable, Optional

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from sklearn.linear_model import HuberRegressor

import jupyter_black

pd.set_option("display.max_columns", None)
jupyter_black.load()

In [2]:
EPS = 1e-12
MISSING_SCORE = -2
MINIMUM_TOTAL_SCORE = 0.545

SYSTEM_SCORE = {
    "Windows": 0.0,
    "MacOS": 0.5,
    "Linux": 1.0,
    "NoOS": 1.0,
}

CONDITION_SCORE = {
    "new": 0.0,
    "refurbished": -0.35,
    "used": -1.0,
}

In [3]:
def mapping_scorer(
    values: pd.Series,
    baseline: float | None = None,
    invert: bool = False,
    missing_score: float = MISSING_SCORE,
    score_map: dict | None = None,
) -> pd.Series:
    if score_map is None:
        return pd.Series(missing_score, index=values.index, dtype=float)

    normalized = values.astype(str).str.strip().str.lower()
    normalized_map = {str(k).strip().lower(): float(v) for k, v in score_map.items()}

    return normalized.map(normalized_map).fillna(missing_score).astype(float)


condition_scorer = partial(mapping_scorer, score_map=CONDITION_SCORE)
system_scorer = partial(mapping_scorer, score_map=SYSTEM_SCORE)


def parse_brand_reliability(df):
    return df["Brand Reliability"].str.replace("%", "").astype(float) / 100


def delta_scorer(
    values: pd.Series,
    baseline: float,
    invert: bool = False,
    missing_score: float = MISSING_SCORE,
) -> pd.Series:
    numeric_values = pd.to_numeric(values, errors="coerce")
    scores = pd.Series(missing_score, index=values.index, dtype=float)

    valid = numeric_values.notna()
    delta = (numeric_values[valid] - baseline) / (abs(baseline) + EPS)

    if invert:
        delta = -delta

    scores.loc[valid] = np.tanh(delta)
    return scores


def ratio_scorer(
    values: pd.Series,
    baseline: float,
    invert: bool = False,
    missing_score: float = MISSING_SCORE,
) -> pd.Series:
    numeric_values = pd.to_numeric(values, errors="coerce")
    scores = pd.Series(missing_score, index=values.index, dtype=float)

    valid = numeric_values.notna() & (numeric_values > 0)
    log_ratio = np.log(numeric_values[valid] / baseline)

    if invert:
        log_ratio = -log_ratio

    scores.loc[valid] = np.tanh(log_ratio)
    return scores

In [4]:
@dataclass
class FeatureConfig:
    name: str
    raw_data_name: str
    parse_function: Optional[Callable]
    score_function: Callable
    weight: float
    invert: bool
    baseline_value: float

In [5]:
FEATURES = [
    FeatureConfig(
        "CPU Score",
        "CPU Performance",
        None,
        delta_scorer,
        0.15,
        False,
        baseline_value=26852,
    ),
    FeatureConfig(
        "GPU Score",
        "GPU Performance",
        None,
        delta_scorer,
        0.14,
        False,
        baseline_value=11.6,
    ),
    FeatureConfig(
        "VRAM Score",
        "VRAM",
        None,
        ratio_scorer,
        0.12,
        False,
        baseline_value=16,
    ),
    FeatureConfig(
        "RAM Score",
        "RAM",
        None,
        ratio_scorer,
        0.12,
        False,
        baseline_value=64,
    ),
    FeatureConfig(
        "Battery Score",
        "Battery",
        None,
        ratio_scorer,
        0.09,
        False,
        baseline_value=399,
    ),
    FeatureConfig(
        "New Score",
        "New",
        None,
        condition_scorer,
        0.06,
        False,
        baseline_value=0,
    ),
    FeatureConfig(
        "Weight Score",
        "Weight",
        None,
        delta_scorer,
        0.05,
        True,
        baseline_value=1.9,
    ),
    FeatureConfig(
        "Storage Score",
        "Storage",
        None,
        ratio_scorer,
        0.04,
        False,
        baseline_value=2048,
    ),
    FeatureConfig(
        "System Score",
        "System",
        None,
        system_scorer,
        0.03,
        False,
        baseline_value=0,
    ),
    FeatureConfig(
        "Resolution Score",
        "PPI",
        None,
        delta_scorer,
        0.03,
        False,
        baseline_value=283,
    ),
    FeatureConfig(
        "Temperature Score",
        "Temperature",
        None,
        delta_scorer,
        0.15,
        True,
        baseline_value=52,
    ),
    FeatureConfig(
        "Brand Reliability Score",
        "Brand Reliability",
        parse_brand_reliability,
        ratio_scorer,
        0.02,
        True,
        baseline_value=0.021,
    ),
]
print(sum(f.weight for f in FEATURES) - 1.0)

0.0


In [6]:
raw = pd.read_csv("laptops.csv")
raw

,URL,Laptop Name,Brand,Brand Reliability,CPU,CPU Performance,GPU,GPU Performance,VRAM,RAM,Battery,PPI,System,Weight,Storage,Temperature,New,Price
0,https://bestware.com/en/xmg-neo-16-e25.html#pr...,NEO 16,XMG,NaN,Ultra 9 275HX,56009.0,RTX 5090,19.0,24,64,352.0,189.0,Linux,2.650,2000,47.0,new,18073
1,NaN,ASUS VivoBook 15,Asus,3.3%,i3-8145U,3769.0,MX250,1.4,2,8,360.0,142.0,Linux,1.900,256,NaN,new,2200
2,NaN,ThinkPad X1 EXTREME G5,Lenovo,2.1%,i9-12900H,26852.0,3080 Ti,11.6,16,64,399.0,283.0,Windows,1.900,2048,52.0,new,12500
3,NaN,Legion 5 Pro-16,Lenovo,2.1%,Ryzen 5 5600H,16471.0,RTX 3050 4GB Laptop GPU,4.5,4,16,420.0,189.0,Linux,2.450,512,39.9,new,4999
4,NaN,AERO 16 XE5,Gigabyte,4.5%,i7-12700H,25229.0,3070 Ti,10.4,8,32,448.0,283.0,Windows,2.300,2000,49.0,new,10099
5,https://techlord.pl/hp-elitebook-840-14-g10-wo...,EliteBook 840,HP,2%,i7-1355U,14029.0,Iris Xe Graphics G7 96EUs,1.9,0,64,456.0,162.0,Windows,1.360,512,32.8,new,7299
6,https://consumer.huawei.com/pl/laptops/mateboo...,MateBook X Pro,Huawei,1.2%,Ultra 7 155H,24617.0,Intel Arc 8-Core iGPU,3.2,0,16,498.0,264.0,Windows,0.980,1000,43.2,new,6799
7,https://bestware.com/en/schenker-work-17-m23.h...,Schenker,Schenker,NaN,i7-1360P,18465.0,Iris Xe Graphics G7 96EUs,1.9,0,64,594.0,127.0,NoOS,2.279,1000,36.5,new,7613
8,NaN,Huawei Matebook 16,Huawei,1.2%,Ryzen 7 5800H,20615.0,Radeon RX Vega 8 (Ryzen 4000/5000),1.4,0,16,768.0,189.0,Windows,1.990,512,40.2,new,4548
9,https://allegro.pl/oferta/laptop-apple-macbook...,MacBook Pro,Apple,0.60%,M4 Pro 12 CPU,32733.0,16 GPU,7.6,24,48,948.0,254.0,MacOS,1.600,2000,47.3,new,14400


In [7]:
cleaned_data = pd.DataFrame()
cleaned_data["Laptop Name"] = raw["Laptop Name"]

for feature in FEATURES:
    if feature.parse_function:
        cleaned_data[feature.raw_data_name] = feature.parse_function(raw)
    else:
        cleaned_data[feature.raw_data_name] = raw[feature.raw_data_name]

cleaned_data["Price"] = raw["Price"]
cleaned_data

,Laptop Name,CPU Performance,GPU Performance,VRAM,RAM,Battery,New,Weight,Storage,System,PPI,Temperature,Brand Reliability,Price
0,NEO 16,56009.0,19.0,24,64,352.0,new,2.650,2000,Linux,189.0,47.0,NaN,18073
1,ASUS VivoBook 15,3769.0,1.4,2,8,360.0,new,1.900,256,Linux,142.0,NaN,0.033,2200
2,ThinkPad X1 EXTREME G5,26852.0,11.6,16,64,399.0,new,1.900,2048,Windows,283.0,52.0,0.021,12500
3,Legion 5 Pro-16,16471.0,4.5,4,16,420.0,new,2.450,512,Linux,189.0,39.9,0.021,4999
4,AERO 16 XE5,25229.0,10.4,8,32,448.0,new,2.300,2000,Windows,283.0,49.0,0.045,10099
5,EliteBook 840,14029.0,1.9,0,64,456.0,new,1.360,512,Windows,162.0,32.8,0.020,7299
6,MateBook X Pro,24617.0,3.2,0,16,498.0,new,0.980,1000,Windows,264.0,43.2,0.012,6799
7,Schenker,18465.0,1.9,0,64,594.0,new,2.279,1000,NoOS,127.0,36.5,NaN,7613
8,Huawei Matebook 16,20615.0,1.4,0,16,768.0,new,1.990,512,Windows,189.0,40.2,0.012,4548
9,MacBook Pro,32733.0,7.6,24,48,948.0,new,1.600,2000,MacOS,254.0,47.3,0.006,14400


In [8]:
score_dataframe = cleaned_data.copy()
for feature in FEATURES:
    col = feature.raw_data_name
    score_dataframe[feature.name] = feature.score_function(
        cleaned_data[col], feature.baseline_value, feature.invert
    )


score_dataframe["Total Score"] = sum(
    score_dataframe[feature.name] * feature.weight for feature in FEATURES
)
score_dataframe = score_dataframe.sort_values("Total Score", ascending=False)
score_dataframe

,Laptop Name,CPU Performance,GPU Performance,VRAM,RAM,Battery,New,Weight,Storage,System,PPI,Temperature,Brand Reliability,Price,CPU Score,GPU Score,VRAM Score,RAM Score,Battery Score,New Score,Weight Score,Storage Score,System Score,Resolution Score,Temperature Score,Brand Reliability Score,Total Score
14,NEO 16,62369.0,19.0,24,64,570.0,new,2.650,2000,NoOS,189.0,47.9,NaN,18800,0.867453,0.563489,0.384615,0.000000,0.342282,0.00,-0.375437,-0.023712,1.0,-0.320456,0.078683,-2.000000,0.258434
10,MacBook Pro 2023,41259.0,11.2,32,64,912.0,refurbished,1.550,2000,MacOS,254.0,45.9,0.006,12978,0.490360,-0.034469,0.600000,0.000000,0.678689,-0.35,0.182155,-0.023712,0.5,-0.102116,0.116773,0.849057,0.235403
0,NEO 16,56009.0,19.0,24,64,352.0,new,2.650,2000,Linux,189.0,47.0,NaN,18073,0.795355,0.563489,0.384615,0.000000,-0.124678,0.00,-0.375437,-0.023712,1.0,-0.320456,0.095859,-2.000000,0.208169
17,ThinkPad T16g,49127.0,17.9,16,64,564.0,new,2.600,2048,Windows,236.0,43.9,0.021,18270,0.680233,0.495334,0.000000,0.000000,0.332906,0.00,-0.352610,0.000000,0.0,-0.164567,0.154521,-0.000000,0.201954
15,Legion 9i 18IAX10,56009.0,19.0,16,64,336.0,new,3.532,2048,Windows,252.0,43.0,0.021,25742,0.795355,0.563489,0.000000,0.000000,-0.170178,0.00,-0.695715,0.000000,0.0,-0.109105,0.171369,-0.000000,0.170522
16,Titan 18 HX AI Gaming Laptop,57483.0,19.0,16,64,258.0,new,3.600,6000,Windows,245.0,48.2,0.054,27307,0.814661,0.563489,0.000000,0.000000,-0.410325,0.00,-0.713725,0.791299,0.0,-0.133474,0.072947,-0.737265,0.152317
9,MacBook Pro,32733.0,7.6,24,48,948.0,new,1.600,2000,MacOS,254.0,47.3,0.006,14400,0.215579,-0.331780,0.384615,-0.280000,0.699026,0.00,0.156596,-0.023712,0.5,-0.102116,0.090139,0.849057,0.110674
2,ThinkPad X1 EXTREME G5,26852.0,11.6,16,64,399.0,new,1.900,2048,Windows,283.0,52.0,0.021,12500,0.000000,0.000000,0.000000,0.000000,0.000000,0.00,-0.000000,0.000000,0.0,0.000000,-0.000000,-0.000000,0.000000
20,ExpertBook Ultra,34550.0,6.4,0,64,966.0,new,1.095,2000,Windows,243.0,38.6,0.033,11872,0.279079,-0.420481,-2.000000,0.000000,0.708518,0.00,0.400030,-0.023712,0.0,-0.140409,0.252136,-0.423529,-0.149048
4,AERO 16 XE5,25229.0,10.4,8,32,448.0,new,2.300,2000,Windows,283.0,49.0,0.045,10099,-0.060369,-0.103081,-0.600000,-0.600000,0.115317,0.00,-0.207470,-0.023712,0.0,0.000000,0.057628,-0.642336,-0.172633


In [9]:
reasonable_deals = score_dataframe[score_dataframe["Total Score"] > MINIMUM_TOTAL_SCORE]
reasonable_deals

,Laptop Name,CPU Performance,GPU Performance,VRAM,RAM,Battery,New,Weight,Storage,System,PPI,Temperature,Brand Reliability,Price,CPU Score,GPU Score,VRAM Score,RAM Score,Battery Score,New Score,Weight Score,Storage Score,System Score,Resolution Score,Temperature Score,Brand Reliability Score,Total Score
